In [1]:
# Notebook 14: CGAN Metrics Evaluation
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path("../data/processed/FD001")
SYNTH_DIR = Path("../data/synthetic/GAN")

X_real = np.load(DATA_DIR / "X_train.npy")
labels = np.load(DATA_DIR / "labels_train.npy")

synth_X_path = SYNTH_DIR / "FD001" / "synth_X.npy"
synth_labels_path = SYNTH_DIR / "FD001" / "synth_labels.npy"

if not synth_X_path.exists() or not synth_labels_path.exists():
    raise FileNotFoundError("CGAN synthetic data not found.")

X_synth = np.load(synth_X_path)
y_synth = np.load(synth_labels_path)

CLASS_NAMES = ["C0 healthy", "C1 early", "C2 advanced", "C3 imminent"]
FEATURE_COLS = [
    "op1","op2",
    "s2","s3","s4","s7","s8","s9",
    "s11","s12","s13","s14","s15","s17","s20","s21",
]
print(f"Real  : {X_real.shape}  labels: {labels.shape}")
print(f"Synth : {X_synth.shape}  labels: {y_synth.shape}")


Real  : (17731, 30, 16)  labels: (17731,)
Synth : (12801, 30, 16)  labels: (12801,)


In [2]:
# Mask dead sensors
synth_std = X_synth.std(axis=(0, 1))
dead_mask = synth_std < 0.01
dead_indices = [i for i in range(len(FEATURE_COLS)) if dead_mask[i]]

X_real_m = X_real.copy()
X_synth_m = X_synth.copy()
for idx in dead_indices:
    X_real_m[:, :, idx] = 0.0
    X_synth_m[:, :, idx] = 0.0

active_features = [FEATURE_COLS[i] for i in range(len(FEATURE_COLS)) if not dead_mask[i]]
print(f"Masked {len(dead_indices)} dead sensor(s)")


Masked 0 dead sensor(s)


In [3]:
# 1. KS Statistic Test
ks_results = []
for c in range(4):
    real_c = X_real_m[labels == c]
    synth_c = X_synth_m[y_synth == c]
    if len(synth_c) == 0:
        continue
    for f_idx, feat in enumerate(FEATURE_COLS):
        if dead_mask[f_idx]: continue
        stat, pval = ks_2samp(real_c[:, :, f_idx].flatten(), synth_c[:, :, f_idx].flatten())
        ks_results.append({
            "class": f"C{c}", "sensor": feat,
            "ks_stat": round(stat, 4), "p_value": round(pval, 4),
            "pass": pval > 0.05
        })

ks_df = pd.DataFrame(ks_results)
pass_rate = ks_df["pass"].mean() * 100
print(f"KS test overall pass rate: {pass_rate:.1f}%")
print(ks_df.groupby("class")["pass"].mean().apply(lambda x: f"{x*100:.1f}%"))


KS test overall pass rate: 0.0%
class
C1    0.0%
C2    0.0%
C3    0.0%
Name: pass, dtype: object


In [4]:
# 2. Discriminative Score
X_r = X_real_m.reshape(len(X_real_m), -1)
X_s = X_synth_m.reshape(len(X_synth_m), -1)

min_n = min(len(X_r), len(X_s))
idx_r = np.random.choice(len(X_r), min_n, replace=False)
idx_s = np.random.choice(len(X_s), min_n, replace=False)

X_disc = np.concatenate([X_r[idx_r], X_s[idx_s]], axis=0)
y_disc = np.array([0]*min_n + [1]*min_n)
X_disc_sc = StandardScaler().fit_transform(X_disc)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
scores = cross_val_score(clf, X_disc_sc, y_disc, cv=5, scoring="accuracy")

disc_score = scores.mean()
print(f"Discriminative score : {disc_score:.3f} +/- {scores.std():.3f}")


Discriminative score : 0.989 +/- 0.001


In [5]:
# 3. Cross-Sensor Correlation Difference
real_flat = X_real.mean(axis=1)
synth_flat = X_synth.mean(axis=1)

real_corr = np.corrcoef(real_flat.T)
synth_corr = np.corrcoef(synth_flat.T)
diff = np.abs(real_corr - synth_corr)

print(f"Mean correlation diff : {diff.mean():.4f}")
print(f"Max correlation diff  : {diff.max():.4f}")


Mean correlation diff : 0.1333
Max correlation diff  : 0.4356
